# theLook Analytics · 01 · Coorti e canali

Due domande del brief, con un test statistico ciascuna:

1. **Coorti**: i clienti che hanno fatto il primo acquisto nel 2026 tornano a comprare più di quelli del 2025? *(domanda 3)*
2. **Canali**: il canale da cui arriva una sessione dipende dal canale con cui il cliente si è iscritto? Se non dipende, un modello di attribution non ha segnale su cui lavorare. *(premessa alla domanda 1)*

**Dati**: viste `marts` costruite con `sql/01_staging.sql` e `sql/02_marts.sql`, con data di taglio al 30 settembre 2026.

**Come eseguirlo**: in [Google Colab](https://colab.research.google.com), con lo stesso account Google del progetto BigQuery. Scrivi l'ID del progetto nella cella qui sotto, poi *Runtime → Esegui tutto*.

In [ ]:
# ID del progetto Google Cloud: è quello che vedi nel selettore in alto
# nella console di BigQuery (può avere un suffisso numerico).
PROJECT_ID = "scrivi-qui-l-id-del-progetto"

from datetime import date, timedelta

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

# Data di taglio e finestra di osservazione
CUTOFF = date(2026, 9, 30)
WINDOW_DAYS = 90
LAST_FIRST_ORDER = CUTOFF - timedelta(days=WINDOW_DAYS)  # 2 luglio 2026

# Stile coerente con il report Power BI: un accento, il resto in grigio
ACCENT, GREY, INK = "#118DFF", "#B3B3B3", "#252423"
plt.rcParams.update({
    "figure.figsize": (9, 4.5),
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlesize": 13,
    "axes.titleweight": "bold",
    "axes.titlelocation": "left",
    "font.size": 10.5,
})


def num(x, decimals=0):
    """12345.6 -> '12.345,6' (formato italiano)."""
    testo = f"{x:,.{decimals}f}"
    return testo.replace(",", "_").replace(".", ",").replace("_", ".")


def pct(x, decimals=1):
    """0.0423 -> '4,2%'."""
    return num(x * 100, decimals) + "%"


def punti(x, decimals=2):
    """Differenza tra due quote: 0.0275 -> '2,75 punti percentuali'."""
    return num(x * 100, decimals) + " punti percentuali"

In [ ]:
# Accesso a BigQuery con l'account Google (si apre una finestra di consenso)
from google.colab import auth
auth.authenticate_user()

from google.cloud import bigquery
client = bigquery.Client(project=PROJECT_ID)


def run(sql):
    """Esegue una query e restituisce un DataFrame.
    La Storage API è disattivata: in sandbox può non essere disponibile."""
    return client.query(sql).to_dataframe(create_bqstorage_client=False)

## 1 · Le coorti del 2026 ricomprano più di quelle del 2025?

**Metrica**: quota di clienti che fa un secondo ordine entro **90 giorni** dal primo.
Stessa finestra per tutti: entrano solo i clienti con il primo ordine entro il 2 luglio 2026, così anche gli ultimi hanno 90 giorni pieni prima del taglio.

**Ipotesi nulla**: la quota di riacquisto è uguale nei due anni. **Test**: z-test per due proporzioni, con intervallo di confidenza al 95% sulla differenza.

In [ ]:
sql_clienti = f"""
SELECT
  user_id,
  first_order_date,
  second_order_date,
  DATE_DIFF(second_order_date, first_order_date, DAY) AS days_to_second
FROM (
  SELECT
    user_id,
    MIN(IF(user_order_number = 1, order_date, NULL)) AS first_order_date,
    MIN(IF(user_order_number = 2, order_date, NULL)) AS second_order_date
  FROM marts.fct_orders
  WHERE status != 'cancelled'
  GROUP BY user_id
)
WHERE first_order_date BETWEEN '2025-01-01' AND '{LAST_FIRST_ORDER}'
"""

clienti = run(sql_clienti)
clienti["first_order_date"] = pd.to_datetime(clienti["first_order_date"])
print(f"{num(len(clienti))} clienti con il primo ordine tra gennaio 2025 e il {LAST_FIRST_ORDER:%d/%m/%Y}")
clienti.head()

In [ ]:
clienti["anno_coorte"] = clienti["first_order_date"].dt.year
clienti["riacquisto_90"] = clienti["days_to_second"].le(WINDOW_DAYS).fillna(False).astype(bool)

per_anno = (
    clienti.groupby("anno_coorte")["riacquisto_90"]
    .agg(clienti="size", riacquisti="sum", quota="mean")
)
per_anno.style.format({"quota": pct})

In [ ]:
def ztest_due_proporzioni(x1, n1, x2, n2, livello=0.95):
    """z-test per la differenza p2 - p1, con intervallo di confidenza di Wald."""
    p1, p2 = x1 / n1, x2 / n2
    p_pool = (x1 + x2) / (n1 + n2)
    se_pool = np.sqrt(p_pool * (1 - p_pool) * (1 / n1 + 1 / n2))
    z = (p2 - p1) / se_pool
    p_value = 2 * stats.norm.sf(abs(z))
    se_diff = np.sqrt(p1 * (1 - p1) / n1 + p2 * (1 - p2) / n2)
    z_crit = stats.norm.ppf(1 - (1 - livello) / 2)
    diff = p2 - p1
    return {
        "p_2025": p1, "p_2026": p2, "differenza": diff,
        "ic_basso": diff - z_crit * se_diff, "ic_alto": diff + z_crit * se_diff,
        "rapporto": p2 / p1, "z": z, "p_value": p_value,
    }


a, b = per_anno.loc[2025], per_anno.loc[2026]
test_coorti = ztest_due_proporzioni(a["riacquisti"], a["clienti"], b["riacquisti"], b["clienti"])

print(f"Riacquisto entro {WINDOW_DAYS} giorni: 2025 {pct(test_coorti['p_2025'])} · 2026 {pct(test_coorti['p_2026'])}")
print(f"Differenza: {punti(test_coorti['differenza'])} "
      f"(IC 95%: da {punti(test_coorti['ic_basso'])} a {punti(test_coorti['ic_alto'])})")
print(f"Rapporto 2026/2025: {num(test_coorti['rapporto'], 2)}")
print(f"z = {num(test_coorti['z'], 2)}, p-value = {test_coorti['p_value']:.2g}")

In [ ]:
# Curva cumulata: quota di clienti che ha già ricomprato dopo d giorni
giorni = np.arange(0, WINDOW_DAYS + 1)
fig, ax = plt.subplots()
for anno, colore, spessore in [(2025, GREY, 2), (2026, ACCENT, 2.5)]:
    gruppo = clienti.loc[clienti["anno_coorte"] == anno, "days_to_second"]
    # fillna(False): chi non ha ricomprato resta nel denominatore
    curva = [(gruppo <= d).fillna(False).mean() for d in giorni]
    ax.plot(giorni, curva, color=colore, linewidth=spessore, label=f"Primo acquisto nel {anno}")
    ax.annotate(pct(curva[-1]), (giorni[-1], curva[-1]), xytext=(6, 0),
                textcoords="offset points", va="center", color=colore, fontweight="bold")

ax.set_title(
    f"Entro {WINDOW_DAYS} giorni ricompra il {pct(test_coorti['p_2026'])} dei clienti 2026 "
    f"contro il {pct(test_coorti['p_2025'])} del 2025"
)
ax.set_xlabel("Giorni dal primo acquisto")
ax.yaxis.set_major_formatter(lambda y, _: pct(y, 0))
ax.set_xlim(0, WINDOW_DAYS + 12)
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

### Il salto è tra un anno e l'altro, o una crescita continua?

Se la quota sale mese dopo mese in modo regolare, il confronto 2025/2026 fotografa una tendenza, non un cambiamento avvenuto a inizio 2026. È un'informazione utile anche per capire se la crescita viene dal generatore dei dati.

In [ ]:
per_mese = (
    clienti.assign(mese=clienti["first_order_date"].dt.to_period("M").dt.to_timestamp())
    .groupby("mese")["riacquisto_90"]
    .agg(clienti="size", quota="mean")
    .reset_index()
)
# Solo mesi interi: luglio 2026 avrebbe appena due giorni di primi acquisti
fine_mese = per_mese["mese"] + pd.offsets.MonthEnd(0)
per_mese = per_mese[fine_mese <= pd.Timestamp(LAST_FIRST_ORDER)].reset_index(drop=True)

# Pendenza della quota per mese di coorte (regressione lineare semplice)
x = np.arange(len(per_mese))
trend = stats.linregress(x, per_mese["quota"])

fig, ax = plt.subplots()
colori = [ACCENT if m.year == 2026 else GREY for m in per_mese["mese"]]
ax.bar(per_mese["mese"], per_mese["quota"], width=20, color=colori)
ax.plot(per_mese["mese"], trend.intercept + trend.slope * x, color=INK, linewidth=1, linestyle="--",
        label=f"Tendenza: {punti(trend.slope)} al mese")
ax.set_title("Quota di riacquisto entro 90 giorni per mese del primo acquisto")
ax.yaxis.set_major_formatter(lambda y, _: pct(y, 0))
ax.legend(frameon=False, loc="upper left")
plt.tight_layout()
plt.show()

print(f"R² della tendenza lineare: {num(trend.rvalue ** 2, 2)}")
print(f"p-value della pendenza: {trend.pvalue:.2g}")

### Lettura dei risultati

- Entro 90 giorni ricompra il **12,5%** dei clienti con il primo acquisto nel 2026, contro il **7,6%** del 2025: +4,9 punti percentuali, 1,65 volte tanto.
- L'intervallo di confidenza al 95% va da +4,2 a +5,7 punti e il p-value è vicino a zero: con 25.598 clienti la differenza non è dovuta al caso.
- Il confronto mese per mese mostra una **crescita continua**, non un salto a inizio 2026: la quota passa dal 5,5% delle coorti di gennaio 2025 al 15,2% di giugno 2026, circa mezzo punto in più ogni mese (R² 0,83).

**Conclusione.** Le coorti recenti ricomprano di più, ma una crescita così regolare fa pensare a una regola del generatore dei dati più che a un cambiamento nel comportamento dei clienti. In un caso reale, prima di attribuirla a un'iniziativa di marketing, andrebbero cercati i fattori che la spiegano: canale, categoria del primo acquisto, campagne email.

## 2 · Il canale della sessione dipende dal canale di iscrizione?

Per l'attribution serve che il percorso dei clienti tra i canali abbia una struttura: per esempio, chi arriva da Paid Search torna più spesso da Email. Se il canale di ogni sessione è indipendente da quello di iscrizione, i modelli di attribution distribuiscono il merito in proporzione ai volumi, senza dire nulla di nuovo.

**Ipotesi nulla**: canale di iscrizione e canale di sessione sono indipendenti. **Test**: chi-quadro di indipendenza. Con centinaia di migliaia di sessioni anche differenze minime risultano "significative", quindi conta soprattutto la **V di Cramér**, che misura la forza del legame da 0 (nessun legame) a 1.

In [ ]:
sql_canali = """
SELECT
  u.signup_channel  AS canale_iscrizione,
  s.session_channel AS canale_sessione,
  COUNT(*)          AS sessioni
FROM marts.fct_sessions AS s
JOIN marts.dim_users    AS u
  ON s.user_id = u.user_id
GROUP BY canale_iscrizione, canale_sessione
"""

canali = run(sql_canali)
# BigQuery restituisce interi "nullable" (Int64): scipy vuole interi normali
canali["sessioni"] = canali["sessioni"].astype("int64")
tabella = canali.pivot_table(index="canale_iscrizione", columns="canale_sessione",
                             values="sessioni", aggfunc="sum", fill_value=0).astype("int64")

# Quote per riga: se le righe sono quasi uguali, i canali sono indipendenti
quote_riga = tabella.div(tabella.sum(axis=1), axis=0)
quote_riga.style.format(pct).background_gradient(cmap="Blues", axis=None)

In [ ]:
chi2, p_value, gradi_liberta, attesi = stats.chi2_contingency(tabella)
n = tabella.to_numpy().sum()
v_cramer = np.sqrt(chi2 / (n * (min(tabella.shape) - 1)))

if v_cramer < 0.1:
    forza = "trascurabile"
elif v_cramer < 0.3:
    forza = "debole"
elif v_cramer < 0.5:
    forza = "moderato"
else:
    forza = "forte"

print(f"Sessioni analizzate: {num(n)}")
print(f"Chi-quadro = {num(chi2, 1)} con {gradi_liberta} gradi di libertà, p-value = {p_value:.2g}")
print(f"V di Cramér = {num(v_cramer, 3)}: legame {forza}")

### Lettura dei risultati

- Su 164.463 sessioni di clienti registrati, la ripartizione tra i canali di sessione è la stessa qualunque sia il canale di iscrizione.
- Chi-quadro 22,4 con 16 gradi di libertà, p-value 0,13: non si può rifiutare l'ipotesi di indipendenza.
- V di Cramér 0,006: anche la forza del legame è praticamente nulla.

**Conclusione.** In questi dati il percorso dei clienti tra i canali non ha struttura. Un modello di attribution multi-touch distribuirebbe il merito in proporzione al traffico, senza dire nulla di nuovo. Per rispondere alla domanda 1 del brief serve un esperimento, per esempio un gruppo di controllo senza campagne, oppure un dataset con percorsi reali.

## Limiti

- I dati di theLook sono sintetici: un risultato può riflettere le regole del generatore più che il comportamento dei clienti.
- La finestra di 90 giorni esclude dal confronto i clienti con il primo ordine dopo il 2 luglio 2026.
- Il 2026 copre solo il primo semestre, il 2025 l'anno intero: il confronto mese per mese della parte 1 serve anche a escludere un effetto stagionale.
- Il riacquisto conta solo ordini non cancellati; un ordine poi reso conta comunque come riacquisto.